# Agent 3 · Suivi de trajectoire

Apprentissage par renforcement à partir des points de guidage et de la vitesse. Le banc isolé fonctionne hors jeu.

In [ ]:
import time
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

def entree_pilote(plan, vitesse_ms, lacet_rads):
    # Les ordonnées des sept points sont fixées par le contrat du plan.
    v = np.r_[2*plan.xy[:,0]-1, plan.cible_kmh/50, vitesse_ms/30, lacet_rads/3]
    if v.shape!=(10,) or not np.isfinite(v).all():
        raise ValueError("Trajectoire ou télémétrie invalide pour le pilote")
    return np.clip(v,-5,5).astype(np.float32)

def pedales(action):
    direction, acceleration = np.asarray(action,dtype=np.float32)
    if not np.isfinite([direction,acceleration]).all():
        raise ValueError("Action du pilote invalide")
    direction, acceleration = np.clip([direction,acceleration],-1,1)
    return np.array([direction,max(0.,acceleration),max(0.,-acceleration)], np.float32)

class SuiviTrajectoire(gym.Env):
    """Adaptateur PPO. Le pilote reçoit exclusivement le plan et l'état dynamique."""
    def __init__(self, core, perception, planificateur):
        self.core = core
        self.perception = perception
        self.planificateur = planificateur
        self.observation_space = spaces.Box(-5.,5.,(10,),np.float32)
        self.action_space = spaces.Box(-1.,1.,(2,),np.float32)

    def observation(self):
        _,p = self.perception.observer(self.core.image_guidage)
        raw = self.core.raw
        self.plan = self.planificateur.proposer(p,raw['speed'],raw['angular_velocity_y'])
        return entree_pilote(self.plan,raw['speed'],raw['angular_velocity_y'])

    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        _,info = self.core.reset(seed=seed,options=options)
        try:
            return self.observation(),info
        except Exception:
            self.core.safe_stop()
            raise

    def step(self, action):
        stamp = float(self.core.raw['timestamp_unix'] if 'timestamp_unix' in self.core.raw else self.core.raw['_received_unix'])
        if not np.isfinite(stamp) or not -.15 <= time.time()-stamp <= .55:
            self.core.safe_stop()
            raise RuntimeError("Plan périmé avant la commande")
        cible = self.plan.cible_kmh
        _,reward,term,trunc,info = self.core.step(pedales(action))
        try:
            obs = self.observation()
        except GuidageIndisponible as exc:
            self.core.safe_stop()
            info = dict(info,guidage_perdu=str(exc),milestone_reached=False,lap_finished=False)
            return np.zeros(10,np.float32),-100.,True,False,info
        except Exception:
            self.core.safe_stop()
            raise
        # Frais de suivi par pas, en plus des récompenses physiques du simulateur.
        ecart = abs(float(obs[0]))
        v_kmh = 3.6*self.core.raw['speed']
        erreur_v = abs(v_kmh-cible)/50
        exces_v = max(0.,v_kmh-cible)/50
        surplus = min(1.,max(0.,v_kmh-cible)/8.)
        pen_cmd = 2.0*surplus*(float(np.clip(action[1],-1.,1.))+.5)
        return obs,float(reward-1.5*ecart-.4*erreur_v-3.0*exces_v-pen_cmd),term,trunc,dict(info,
            ecart_image_normalise=ecart, vitesse_cible_kmh=cible)

    def close(self):
        self.core.close()

class FreinPendantMiseAJour(BaseCallback):
    def __init__(self, core, archiver):
        super().__init__()
        self.core = core
        self.archiver = archiver
        self.pause = False

    def _on_rollout_start(self):
        if self.pause:
            self.archiver(self.model)
            self.model._last_obs = self.training_env.reset()
            self.model._last_episode_starts = np.ones(1,dtype=bool)
        self.core.begin_rollout(self.model.n_steps)

    def _on_step(self):
        return True

    def _on_rollout_end(self):
        self.core.safe_stop()
        self.pause = True

def nouveau_pilote(env, seed=74, n_steps=128):
    return PPO('MlpPolicy',env,seed=seed,n_steps=n_steps,batch_size=32,n_epochs=4,
               policy_kwargs=dict(net_arch=[64,64]),device='cpu',verbose=0)

In [ ]:
from types import SimpleNamespace

class BancSuivi(gym.Env):
    """Petit banc latéral. Le plan oracle est fourni en coordonnées image."""
    def __init__(self):
        self.action_space = spaces.Box(-1.,1.,(2,),np.float32)
        self.observation_space = spaces.Box(-5.,5.,(10,),np.float32)

    def observer(self):
        c = self.virage*np.linspace(0,1,7)**2
        x = np.clip(.5 + self.erreur + c,.01,.99)
        plan = SimpleNamespace(xy=np.column_stack((x,np.linspace(.87,.57,7))),cible_kmh=28.)
        return entree_pilote(plan,self.vitesse,self.rotation)

    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        self.erreur = self.np_random.uniform(-.18,.18)
        self.virage = self.np_random.uniform(-.2,.2)
        self.vitesse = self.np_random.uniform(2,8)
        self.i = 0
        self.rotation = 0.
        return self.observer(),{}

    def step(self,action):
        steer,gaz,frein = pedales(action)
        self.vitesse = np.clip(self.vitesse+.25*gaz-.45*frein-.025,0,16.)
        self.rotation = .75*self.rotation + .25*steer
        self.erreur += (self.virage-self.rotation)*self.vitesse*.003
        self.i += 1
        sortie = bool(abs(self.erreur)>.45)
        cout = 3*abs(self.erreur)+abs(self.vitesse*3.6-28)/28
        return self.observer(),float(1-cout-8*sortie),sortie,self.i>=80,{'ecart':abs(self.erreur)}

def tester_a3():
    import tempfile
    env = BancSuivi()
    modele = nouveau_pilote(env,n_steps=64)
    modele.learn(512)
    assert modele.num_timesteps==512
    with tempfile.TemporaryDirectory() as d:
        fichier = Path(d)/'pilote.zip'
        modele.save(fichier)
        relu = PPO.load(fichier,device='cpu')
        obs,_ = env.reset(seed=987)
        assert np.allclose(modele.predict(obs,deterministic=True)[0],relu.predict(obs,deterministic=True)[0])
        sorties = 0
        erreurs = []
        for seed in range(3):
            obs,_ = env.reset(seed=333+seed)
            for _ in range(80):
                action,_ = relu.predict(obs,deterministic=True)
                volant,gaz,frein = pedales(action)
                assert -1<=volant<=1 and gaz*frein==0
                obs,_,term,trunc,info = env.step(action)
                erreurs.append(info['ecart'])
                if term or trunc:
                    sorties += int(term)
                    break
    ident = archiver('pilote','.zip',modele.save,dict(domaine='banc',perception=None,
                     trajectoire=None,pas=512,stage=0,parent=None))
    return dict(version_a3=ident,pas_entrainement_banc=512,episodes_banc=3,sorties_banc=sorties,
                ecart_image_moyen=float(np.mean(erreurs)))

In [ ]:
def tester_integration(a1,a2):
    # Remplace seulement le simulateur et vJoy. Les trois agents sont réels.
    class CircuitMemoire:
        def __init__(self):
            self.commandes = []
            self.freine = False
            self.raw = {'speed':6.,'angular_velocity_y':0.,'timestamp_unix':time.time()}
        def reset(self,**options):
            self.image_guidage = image.copy()
            return None,{}
        def step(self,action):
            self.raw['timestamp_unix'] = time.time()
            self.commandes.append(action.copy())
            return None,1.,False,False,{}
        def safe_stop(self):
            self.freine = True
        def close(self):
            self.safe_stop()
    rng = np.random.default_rng(564)
    for _ in range(40):
        image,_ = scene_synthetique(rng)
        try:
            plan = a2.proposer(a1.observer(image)[1],6.,0.)
            break
        except GuidageIndisponible:
            continue
    else:
        raise AssertionError("Aucun plan de référence")
    core = CircuitMemoire()
    env = SuiviTrajectoire(core,a1,a2)
    try:
        obs,_ = env.reset()
        assert np.allclose(obs,entree_pilote(plan,6.,0.))
        if VERSION_A3:
            fichier,meta = relire('pilote',VERSION_A3)
            if meta.get('domaine')!='banc' and (meta['perception'],meta['trajectoire'])!=(VERSION_A1,VERSION_A2):
                raise ValueError("Versions de guidage incompatibles avec ce pilote")
            modele = PPO.load(fichier,env=env,device='cpu')
        else:
            modele = nouveau_pilote(env,n_steps=32)
        obs,_ = env.reset()
        core.raw['timestamp_unix'] = time.time()
        action,_ = modele.predict(obs,deterministic=True)
        _,r,term,trunc,info = env.step(action)
        assert np.array_equal(core.commandes[-1],pedales(action)) and np.isfinite(r)
        assert not term and not trunc and info['vitesse_cible_kmh']==plan.cible_kmh
        class SansPiste:
            def observer(self,image):
                return None,np.zeros((96,128),np.float32)
        env.perception = SansPiste()
        _,_,term,_,info = env.step(np.zeros(2,np.float32))
        assert term and core.freine and info['guidage_perdu']
    finally:
        env.close()
    return dict(commandes_simulees=len(core.commandes),frein_sur_perte=True)